# 17 — Likelihood and Information Theory

**Master syllabus coverage:** V2 2.8

## Why this matters

Next-token training is maximum likelihood expressed as cross-entropy. Entropy, KL divergence, and perplexity provide the language needed to interpret that objective and its evaluation.

## Learning objectives

- Explain why sequence likelihood is computed in log space.
- Calculate self-information and entropy.
- Verify the relationship among entropy, cross-entropy, and KL divergence.
- Convert mean token NLL into perplexity with proper caveats.

Work through the notebook in order. Predict shapes and results before running each code cell. An assertion failure is part of the lesson: read it before changing code.


## 1. Likelihood scores parameters using observed data

Probability treats parameters as fixed and asks about outcomes. Likelihood treats the
observed outcomes as fixed and compares parameter settings. For independent tokens:

$$\mathcal{L}(\theta)=\prod_t p_\theta(x_t\mid x_{<t}),\qquad
\log\mathcal{L}(\theta)=\sum_t\log p_\theta(x_t\mid x_{<t}).$$

Logs turn products into sums and prevent extremely small products from underflowing.


In [ ]:
import numpy as np
import torch

probabilities = np.array([0.1] * 200, dtype=np.float32)
product = probabilities.prod()
log_likelihood = np.log(probabilities.astype(np.float64)).sum()
print("float32 product:", product)
print("log likelihood:", log_likelihood)
assert product == 0.0 and np.isfinite(log_likelihood)


## 2. Entropy is expected surprise

Self-information is $I(x)=-\log p(x)$. Entropy averages surprise under distribution
$p$:

$$H(p)=-\sum_i p_i\log p_i.$$

Natural logs measure **nats**; base-2 logs measure **bits**. A deterministic categorical
distribution has zero entropy; a uniform $V$-class distribution has maximum entropy
$\log V$.


In [ ]:
def entropy(p: np.ndarray) -> float:
    p = np.asarray(p, dtype=np.float64)
    positive = p > 0
    return float(-np.sum(p[positive] * np.log(p[positive])))

for p in (np.array([1.0, 0.0, 0.0]), np.array([0.8, 0.1, 0.1]), np.ones(3) / 3):
    print(p, "entropy=", entropy(p))
assert np.isclose(entropy(np.ones(3) / 3), np.log(3))


## 3. Cross-entropy and KL divergence

$$H(p,q)=-\sum_i p_i\log q_i,\qquad
D_{KL}(p\|q)=\sum_i p_i\log\frac{p_i}{q_i}.$$

Their identity is $H(p,q)=H(p)+D_{KL}(p\|q)$. Since $H(p)$ does not depend on model
$q$, minimizing cross-entropy fits $q$ toward data distribution $p$. KL is asymmetric
and is not a metric.


In [ ]:
def cross_entropy(p: np.ndarray, q: np.ndarray) -> float:
    if np.any((p > 0) & (q <= 0)):
        return float("inf")
    positive = p > 0
    return float(-np.sum(p[positive] * np.log(q[positive])))

p = np.array([0.7, 0.2, 0.1])
q = np.array([0.5, 0.4, 0.1])
kl_pq = cross_entropy(p, q) - entropy(p)
kl_qp = cross_entropy(q, p) - entropy(q)
print("H(p):", entropy(p), "H(p,q):", cross_entropy(p, q), "KL(p||q):", kl_pq)
assert kl_pq >= 0 and not np.isclose(kl_pq, kl_qp)


## 4. Token loss and perplexity

With a one-hot target token $y$, cross-entropy is simply $-\log q_y$. Mean token NLL
exponentiates to perplexity: $\operatorname{PPL}=\exp(\text{mean NLL})$. Roughly, it is
the effective number of equally likely choices under the model—but comparisons require
the same tokenizer and evaluation protocol.


In [ ]:
token_probabilities = torch.tensor([0.5, 0.25, 0.125, 0.125], dtype=torch.float64)
targets = torch.tensor([0, 1, 2, 3])
losses = -torch.log(token_probabilities[targets])
mean_nll = losses.mean()
perplexity = mean_nll.exp()
print("token losses:", losses.tolist())
print("mean NLL:", mean_nll.item(), "perplexity:", perplexity.item())
assert perplexity.item() >= 1.0


## Failure modes to recognize

- **Probability product underflow:** a valid sequence receives numeric zero likelihood.
- **Log of zero:** impossible model events produce infinite loss.
- **KL symmetry assumption:** reversing arguments changes the question and result.
- **Tokenizer-blind perplexity:** scores with different token units are compared directly.

These are diagnostic patterns, not trivia. When a future model fails, reduce the problem to the smallest example in this notebook and test the relevant invariant.


## Deliberate practice

1. Compute entropy in bits and nats for the same distribution and explain the unit conversion.
2. Find a q distribution that gives infinite cross-entropy under a chosen p and explain why.
3. Show that uniform predictions over V classes produce perplexity V.

Do not merely rerun the provided cells. Make a prediction, change one variable, and write down why the result did or did not match your prediction.

## Exit ticket

**You are ready to continue when:** you can derive next-token NLL from maximum likelihood and explain entropy, cross-entropy, KL, and perplexity precisely.

**Next:** 18 — Softmax, cross-entropy, and numerical stability.
